# Retail Credit Default Early Warning

## Interactive exploration

This Snowflake Workspace notebook investigates the  account portfolio data. The target is whether an existing account reaches the default state within 90 days.

The intended use is portfolio monitoring and case prioritisation.

## 1. Create Snowpark session and execution context

In [ ]:
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.use_role("CRISK_DEMO_DEVELOPER")
session.use_warehouse("CRISK_DEMO_WH")
session.use_database("CRISK_DEMO_DB")

Check context using SQL

In [ ]:
%%sql -r execution_context
SELECT
  CURRENT_ROLE() AS ACTIVE_ROLE,
  CURRENT_WAREHOUSE() AS ACTIVE_WAREHOUSE,
  CURRENT_DATABASE() AS ACTIVE_DATABASE,
  CURRENT_SCHEMA() AS ACTIVE_SCHEMA;

## 2. Inspect the data contract before choosing features

The training relation has one row per account and observation month with finalised ground truth. Start by examining its schema and a bounded sample rather than assuming which columns should enter a model.

In [ ]:
%%sql -r training_schema
DESCRIBE VIEW CRISK_DEMO_DB.RAW.TRAINING_BASE;

In [ ]:
traning_base_pd =  pd.read_snowflake("CRISK_DEMO_DB.RAW.TRAINING_BASE")

traning_base_pd.sort_values(["OBSERVATION_DATE", "ACCOUNT_ID"]).head(20)

The sample should show operational account attributes, behavioural measures, a finalised 90-day outcome, and its finality date. Account ID, observation date, and outcome finality date describe identity or timing; they are not automatically modelling features.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*materialization.*")

training_row_count = len(traning_base_pd)
account_count = traning_base_pd["ACCOUNT_ID"].nunique()
first_observation = traning_base_pd["OBSERVATION_DATE"].min()
last_observation = traning_base_pd["OBSERVATION_DATE"].max()
observation_month_count = traning_base_pd["OBSERVATION_DATE"].nunique()
duplicate_key_count = training_row_count - traning_base_pd.groupby(["ACCOUNT_ID", "OBSERVATION_DATE"]).ngroups
non_final_label_count = (traning_base_pd["OUTCOME_FINALITY_DATE"] > pd.Timestamp("2026-09-01")).sum()

contract_summary = pd.DataFrame({
    "TRAINING_ROW_COUNT": [training_row_count],
    "ACCOUNT_COUNT": [account_count],
    "FIRST_OBSERVATION": [first_observation],
    "LAST_OBSERVATION": [last_observation],
    "OBSERVATION_MONTH_COUNT": [observation_month_count],
    "DUPLICATE_KEY_COUNT": [duplicate_key_count],
    "NON_FINAL_LABEL_COUNT": [non_final_label_count],
})
contract_summary

In [ ]:
null_counts = traning_base_pd.isnull().sum()
null_profile = null_counts.to_frame(name="NULL_COUNT").T
null_profile.columns = [f"{col}_NULLS" for col in null_profile.columns]
null_profile

A valid Phase 2 starting point has no duplicate account-date keys, no non-final outcomes in `TRAINING_BASE`, and no unexplained missing values. Any exception should stop modelling and be resolved in the data contract.

## 3. Understand target availability and time

Credit outcomes arrive after the 90-day window. The full outcome table therefore contains both finalised historical labels and pending recent observations. Training must use only the finalised relation.

In [ ]:
default_outcome_pd = pd.read_snowflake("CRISK_DEMO_DB.RAW.DEFAULT_OUTCOME")

outcome_stats = default_outcome_pd.groupby("OUTCOME_STATUS").agg(
    OUTCOME_COUNT=("OUTCOME_STATUS", "count"),
    FIRST_OBSERVATION=("OBSERVATION_DATE", "min"),
    LAST_OBSERVATION=("OBSERVATION_DATE", "max"),
).reset_index()

finalised_pre = default_outcome_pd[
    (default_outcome_pd["OUTCOME_STATUS"] == "FINALISED") &
    (default_outcome_pd["OBSERVATION_DATE"] < pd.Timestamp("2026-01-01"))
]
pre_holdout_rate = pd.DataFrame({
    "OUTCOME_STATUS": ["FINALISED"],
    "PRE_HOLDOUT_DEFAULT_RATE": [finalised_pre["DEFAULT_WITHIN_90D"].mean()],
})
outcome_stats = outcome_stats.merge(pre_holdout_rate, on="OUTCOME_STATUS", how="left")
outcome_stats.sort_values("OUTCOME_STATUS")

In [ ]:
pre_holdout = traning_base_pd[traning_base_pd["OBSERVATION_DATE"] < pd.Timestamp("2026-01-01")]

monthly_target = pre_holdout.groupby("OBSERVATION_DATE").agg(
    TRAINING_ROW_COUNT=("OBSERVATION_DATE", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("OBSERVATION_DATE")
monthly_target

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

monthly_target_plot = monthly_target.copy()
monthly_target_plot["OBSERVATION_DATE"] = pd.to_datetime(monthly_target_plot["OBSERVATION_DATE"])

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
sns.lineplot(data=monthly_target_plot, x="OBSERVATION_DATE", y="DEFAULT_RATE", marker="o", ax=axes[0])
axes[0].set_title("Pre-holdout 90-day default rate by observation month")
axes[0].set_ylabel("Default rate")

sns.barplot(data=monthly_target_plot, x="OBSERVATION_DATE", y="TRAINING_ROW_COUNT", color="steelblue", ax=axes[1])
axes[1].set_title("Pre-holdout training observations by month")
axes[1].set_xlabel("Observation month")
axes[1].set_ylabel("Training rows")
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

The pre-holdout monthly series reveals two modelling constraints: the target is imbalanced and observations are ordered in time. Random row splitting would place the same accounts and adjacent months on both sides of the split, producing optimistic evidence. Outcomes from 2026 onward remain untouched until final Phase 3 evaluation.

## 4. Discover candidate features from the observed schema

Classify the available columns only after inspecting the schema. This makes exclusions and candidate roles visible rather than embedding a preselected list inside training code.

In [ ]:
dtypes_series = traning_base_pd.dtypes
col_info = []
for i, (col, dtype) in enumerate(dtypes_series.items()):
    dtype_str = str(dtype)
    if col == "ACCOUNT_ID":
        role = "IDENTIFIER_EXCLUDE"
    elif col in ("OBSERVATION_DATE", "OUTCOME_FINALITY_DATE"):
        role = "TIME_CONTROL_EXCLUDE"
    elif col == "DEFAULT_WITHIN_90D":
        role = "TARGET"
    elif "object" in dtype_str or "string" in dtype_str:
        role = "CATEGORICAL_CANDIDATE"
    else:
        role = "NUMERIC_CANDIDATE"
    col_info.append({"COLUMN_NAME": col, "DATA_TYPE": dtype_str, "ORDINAL_POSITION": i + 1, "PROVISIONAL_ROLE": role})

column_roles = pd.DataFrame(col_info)
column_roles

The provisional roles are a review aid, not the final feature contract. Identifier and outcome-timing fields are excluded to prevent memorisation and leakage. Observation date controls temporal splitting but is not passed directly to the initial model.

## 5. Inspect numeric distributions

The following table exposes scale, centre, tails, and operational bounds before transformations are chosen.

In [ ]:
numeric_cols = [
    "ACCOUNT_AGE_MONTHS", "CREDIT_LIMIT", "UTILISATION_RATIO",
    "MONTHLY_INCOME_ESTIMATE", "PAYMENT_AMOUNT_30D",
    "MISSED_PAYMENTS_6M", "CUSTOMER_CONTACTS_90D",
]
numeric_profile = traning_base_pd[numeric_cols].agg(["min", "mean", "median", "max"]).T
numeric_profile.columns = ["MINIMUM", "MEAN", "MEDIAN", "MAXIMUM"]
numeric_profile.index.name = "FEATURE"
numeric_profile = numeric_profile.reset_index().sort_values("FEATURE")
numeric_profile

In [ ]:
target_profile = pre_holdout.groupby("DEFAULT_WITHIN_90D").agg(
    OBSERVATION_COUNT=("DEFAULT_WITHIN_90D", "count"),
    AVG_UTILISATION_RATIO=("UTILISATION_RATIO", "mean"),
    AVG_MONTHLY_INCOME=("MONTHLY_INCOME_ESTIMATE", "mean"),
    AVG_PAYMENT_AMOUNT_30D=("PAYMENT_AMOUNT_30D", "mean"),
    MISSED_PAYMENT_RATE_30D=("MISSED_PAYMENT_30D", "mean"),
    AVG_MISSED_PAYMENTS_6M=("MISSED_PAYMENTS_6M", "mean"),
    ARREARS_RATE_30D=("IN_ARREARS_30D", "mean"),
    AVG_CUSTOMER_CONTACTS_90D=("CUSTOMER_CONTACTS_90D", "mean"),
).reset_index().sort_values("DEFAULT_WITHIN_90D")
target_profile

In [ ]:
import numpy as np

util_data = pre_holdout.copy()
util_data["UTILISATION_BIN"] = (util_data["UTILISATION_RATIO"] * 10 // 1) / 10
utilisation_distribution = util_data.groupby(["UTILISATION_BIN", "DEFAULT_WITHIN_90D"]).agg(
    OBSERVATION_COUNT=("DEFAULT_WITHIN_90D", "count"),
).reset_index().sort_values(["UTILISATION_BIN", "DEFAULT_WITHIN_90D"])
utilisation_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=utilisation_distribution,
    x="UTILISATION_BIN",
    y="OBSERVATION_COUNT",
    hue="DEFAULT_WITHIN_90D",
    ax=ax,
)
ax.set_title("Utilisation distribution by finalised outcome")
ax.set_xlabel("Utilisation ratio bin")
ax.set_ylabel("Account-month observations")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

Differences between outcome classes indicate useful signal but do not establish causality. The logarithmic count axis keeps the minority default class visible. Scaling and transformation decisions remain deferred until the modelling pipeline is chosen.

## 6. Inspect categorical support and outcome rates

Operational segments can be modelling candidates and later monitoring slices. Small groups or unstable rates require caution even when aggregate performance is strong.

In [ ]:
product_outcomes = pre_holdout.groupby("PRODUCT_TYPE").agg(
    OBSERVATION_COUNT=("PRODUCT_TYPE", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("DEFAULT_RATE", ascending=False)
product_outcomes

In [ ]:
channel_outcomes = pre_holdout.groupby("ORIGINATION_CHANNEL").agg(
    OBSERVATION_COUNT=("ORIGINATION_CHANNEL", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("DEFAULT_RATE", ascending=False)
channel_outcomes

In [ ]:
tenure_outcomes = pre_holdout.groupby("TENURE_BAND").agg(
    OBSERVATION_COUNT=("TENURE_BAND", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("DEFAULT_RATE", ascending=False)
tenure_outcomes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for data, category, title, axis in (
    (product_outcomes, "PRODUCT_TYPE", "Product", axes[0]),
    (channel_outcomes, "ORIGINATION_CHANNEL", "Origination channel", axes[1]),
    (tenure_outcomes, "TENURE_BAND", "Tenure", axes[2]),
):
    sns.barplot(data=data, x=category, y="DEFAULT_RATE", ax=axis, color="steelblue")
    axis.set_title(f"Default rate by {title.lower()}")
    axis.set_xlabel(title)
    axis.set_ylabel("Finalised default rate")
    axis.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 7. Examine feature stability before the held-out period

A feature can be predictive yet operationally fragile. Compare development with the later validation period while leaving observations from 2026 onward untouched for final evaluation.

In [ ]:
scenario_data = pre_holdout.copy()
scenario_data["ANALYSIS_PERIOD"] = "DEVELOPMENT"
scenario_data.loc[scenario_data["OBSERVATION_DATE"] >= pd.Timestamp("2025-07-01"), "ANALYSIS_PERIOD"] = "VALIDATION"

scenario_profile = scenario_data.groupby("ANALYSIS_PERIOD").agg(
    OBSERVATION_COUNT=("ANALYSIS_PERIOD", "count"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
    AVG_UTILISATION_RATIO=("UTILISATION_RATIO", "mean"),
    AVG_PAYMENT_AMOUNT_30D=("PAYMENT_AMOUNT_30D", "mean"),
    MISSED_PAYMENT_RATE_30D=("MISSED_PAYMENT_30D", "mean"),
    AVG_MISSED_PAYMENTS_6M=("MISSED_PAYMENTS_6M", "mean"),
    ARREARS_RATE_30D=("IN_ARREARS_30D", "mean"),
    AVG_CUSTOMER_CONTACTS_90D=("CUSTOMER_CONTACTS_90D", "mean"),
).reset_index().sort_values("ANALYSIS_PERIOD")
scenario_profile

In [ ]:
scenario_long = scenario_profile.melt(
    id_vars=["ANALYSIS_PERIOD"],
    value_vars=[
        "AVG_UTILISATION_RATIO",
        "MISSED_PAYMENT_RATE_30D",
        "AVG_MISSED_PAYMENTS_6M",
        "ARREARS_RATE_30D",
    ],
    var_name="MEASURE",
    value_name="VALUE",
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=scenario_long, x="MEASURE", y="VALUE", hue="ANALYSIS_PERIOD", ax=ax)
ax.set_title("Feature behaviour across development and validation")
ax.set_xlabel("Measure")
ax.set_ylabel("Mean or rate")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

Differences between development and validation provide an early stability check without exposing the held-out target. The controlled drift period begins in 2026 and remains reserved for Phase 3 final evaluation.

## 8. Define temporal development, validation, and held-out windows

Use contiguous observation periods. Development ends before validation, and the held-out test begins at the controlled drift boundary. No model or feature decision should use held-out outcomes.

In [ ]:
splits = traning_base_pd.copy()
splits["SPLIT_NAME"] = "HELD_OUT_TEST"
splits.loc[splits["OBSERVATION_DATE"] < pd.Timestamp("2026-01-01"), "SPLIT_NAME"] = "VALIDATION"
splits.loc[splits["OBSERVATION_DATE"] < pd.Timestamp("2025-07-01"), "SPLIT_NAME"] = "DEVELOPMENT"

split_stats = splits.groupby("SPLIT_NAME").agg(
    FIRST_OBSERVATION=("OBSERVATION_DATE", "min"),
    LAST_OBSERVATION=("OBSERVATION_DATE", "max"),
    OBSERVATION_COUNT=("SPLIT_NAME", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
).reset_index()

visible = splits[splits["SPLIT_NAME"] != "HELD_OUT_TEST"].groupby("SPLIT_NAME").agg(
    VISIBLE_DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    VISIBLE_DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index()

split_assignment = split_stats.merge(visible, on="SPLIT_NAME", how="left")
split_assignment.sort_values("FIRST_OBSERVATION")

The split is provisional until its row counts and target support are reviewed. Repeated accounts across periods are intentional for forward-looking evaluation, but no future row from an account can influence fitting for an earlier scoring date. Hyperparameter and threshold decisions use development and validation only.

## 9. Establish baselines and evaluation intent

Accuracy is not an informative primary metric for an imbalanced target. The operational baseline is the historical prevalence and the practical capacity constraint is reviewing the highest-risk 10% of scored accounts.

In [ ]:
baseline = splits.groupby("SPLIT_NAME").agg(
    OBSERVATION_COUNT=("SPLIT_NAME", "count"),
).reset_index()

visible_prevalence = splits[splits["SPLIT_NAME"] != "HELD_OUT_TEST"].groupby("SPLIT_NAME").agg(
    VISIBLE_PREVALENCE_BASELINE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index()

baseline = baseline.merge(visible_prevalence, on="SPLIT_NAME", how="left")
baseline["TARGET_REVIEW_RATE"] = 0.10
baseline["TARGET_REVIEW_COUNT"] = np.ceil(baseline["OBSERVATION_COUNT"] * 0.10).astype(int)
baseline.sort_values("SPLIT_NAME")

Phase 3 will compare candidates using:

- ROC AUC for ranking across thresholds.
- Average precision for minority-class ranking quality.
- Brier score and calibration evidence for probability quality.
- Recall among the highest-risk 10% of accounts for review-capacity relevance.
- Segment ROC AUC and support for product, origination channel, and tenure.

Configured candidate gates are ROC AUC ≥ 0.72, average precision ≥ 0.20, Brier score ≤ 0.16, and segment ROC AUC ≥ 0.62. These are demonstration thresholds and must not be treated as real credit-risk policy.

## 10. Provisional feature decision

**Include for the first candidate:**

- Categorical: `PRODUCT_TYPE`, `ORIGINATION_CHANNEL`, `TENURE_BAND`.
- Numeric: `ACCOUNT_AGE_MONTHS`, `CREDIT_LIMIT`, `UTILISATION_RATIO`, `MONTHLY_INCOME_ESTIMATE`, `PAYMENT_AMOUNT_30D`, `MISSED_PAYMENT_30D`, `MISSED_PAYMENTS_6M`, `IN_ARREARS_30D`, `CUSTOMER_CONTACTS_90D`.

**Exclude from model inputs:**

- `ACCOUNT_ID`: identifier and memorisation risk.
- `OBSERVATION_DATE`: split/control field; avoid learning a synthetic calendar shortcut in the first candidate.
- `OUTCOME_FINALITY_DATE`: label-availability information and direct leakage risk.
- `DEFAULT_WITHIN_90D`: target.

**Questions carried into Phase 3:**

- Do linear and tree-based candidates react differently to the scale and bounded counts?
- Does origination channel add stable signal or only segment variation?
- How much performance changes between validation and controlled-drift held-out data?
- Are probabilities sufficiently calibrated for prioritisation, or is post-fit calibration required?

This phase ends with a reviewable hypothesis. It does not train or register a model.